# Log Anomaly Detection

## Phase 4: Feature Engineering

This phase constructs the two feature representations the project's methodology commits to comparing: a bag-of-events (template count vector) representation for a classical baseline classifier, and an ordered template-sequence representation for the LSTM model. Both are built for all three splits (train, validation, test), since feature construction is a deterministic transformation of already-frozen artifacts (the Drain template dictionary and OOV policy from `02_log_parsing.ipynb`) and fits nothing new; no leakage is introduced by computing test features, only by inspecting them against test labels, which this notebook does not do.

Three constraints carried forward from prior phases govern this notebook:

1. **Test labels remain sealed.** Test-split features are constructed and persisted, but `anomaly_label.csv` is never filtered to or joined with test blocks anywhere in this notebook. Label attachment (Section 4.6) is explicitly scoped to train and validation only.
2. **Block-level censoring is mandatory, for every split, not only train and validation.** `03_eda.ipynb` exercised this rule only for train and validation; this notebook is the first to apply it to test as well, using the same `data/interim/block_lifespans.csv` and `data/interim/split_line_boundaries.json` artifacts (both already cover all 575,061 blocks and all three splits, not only train+val).
3. **No new fitting happens here.** The Drain template dictionary, its OOV token, and the 70/15/15 split itself are all frozen inputs from earlier phases. This notebook only transforms already-parsed data into model-ready arrays; it does not select features, tune thresholds, or fit a scaler. If a classical baseline later requires scaled inputs, that scaler is fit in `05_modeling.ipynb` on train only, following the same fitting discipline used throughout this project, not here.

The seven sub-sections performed here are: loading all required artifacts across the full file (not the train+val-restricted load used in Phase 3), resolving the open truncation-handling policy left by Phase 3, building the per-block censored event base for every split, constructing the bag-of-events feature matrix, constructing the sequence feature artifact, attaching labels to train and validation only, and a phase summary.

### 4.1 Load Full-File Artifacts (All Splits)

Unlike `03_eda.ipynb`, which restricted its load to the fit-eligible (train+val) row range, this notebook needs `data/interim/hdfs_parsed_events.csv` in full, since test features must be built too. `data/interim/block_lifespans.csv` and `data/interim/block_split_assignments.csv` already cover all 575,061 blocks across all three splits and require no re-scoping.

In [2]:
import json
import yaml
import pandas as pd
from pathlib import Path

# 1. Resolve project root and load configuration
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# Resolve paths from config
ASSIGNMENTS_PATH = PROJECT_ROOT / config["paths"]["block_assignments"]
RAW_LABELS_PATH = PROJECT_ROOT / config["paths"]["raw_labels"]
PARSED_EVENTS_PATH = PROJECT_ROOT / config["paths"]["parsed_events"]
BOUNDARIES_PATH = PROJECT_ROOT / config["paths"]["split_boundaries"]
RAW_LOG_PATH = PROJECT_ROOT / config["paths"]["raw_log"]
TEMPLATE_DICT_PATH = PROJECT_ROOT / config["paths"]["template_dict"]
PROCESSED_DIR = PROJECT_ROOT / config["paths"]["processed_dir"]
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

LIFESPANS_PATH = PROJECT_ROOT / config.get("paths", {}).get("block_lifespans", "data/interim/block_lifespans.csv")

# 2. Load block assignments (Full 575,061 blocks across all 3 splits)
assert ASSIGNMENTS_PATH.exists(), f"Missing assignments at {ASSIGNMENTS_PATH}"
df_assignments = pd.read_csv(ASSIGNMENTS_PATH)
total_blocks_count = len(df_assignments)

split_block_counts = df_assignments["split"].value_counts().to_dict()
print("--- Block Split Distribution (All Splits) ---")
for sp in ["train", "val", "test"]:
    print(f"  {sp:<6} blocks: {split_block_counts.get(sp, 0):,}")
print(f"Total blocks mapped : {total_blocks_count:,}")
assert total_blocks_count == 575_061, f"Expected 575,061 blocks, found {total_blocks_count:,}"

# 3. Load split line boundaries
assert BOUNDARIES_PATH.exists(), f"Missing split boundaries at {BOUNDARIES_PATH}"
with open(BOUNDARIES_PATH, "r", encoding="utf-8") as f:
    split_boundaries = json.load(f)

print("\n--- Split Line Boundaries ---")
for sp in ["train", "val", "test"]:
    s_l = split_boundaries[sp]["start_line"]
    e_l = split_boundaries[sp]["end_line"]
    print(f"  {sp:<6} line range : [{s_l:,}, {e_l:,}] ({e_l - s_l + 1:,} lines)")

# 4. Load template dictionary (Confirm 48 templates + OOV schema)
assert TEMPLATE_DICT_PATH.exists(), f"Missing template dictionary at {TEMPLATE_DICT_PATH}"
with open(TEMPLATE_DICT_PATH, "r", encoding="utf-8") as f:
    template_dict = json.load(f)

template_ids = sorted(template_dict.keys(), key=lambda x: int(x[1:]))
oov_token = config["drain"]["oov_token"]
feature_columns = template_ids + [oov_token]
print(f"\nDiscovered templates : {len(template_ids)} ({template_ids[0]} to {template_ids[-1]})")
print(f"OOV token configured : '{oov_token}'")
print(f"Total BoE vocab size : {len(feature_columns)} columns")

# 5. Load block lifespans (Full file)
assert LIFESPANS_PATH.exists(), f"Missing block lifespans at {LIFESPANS_PATH}"
df_lifespans = pd.read_csv(LIFESPANS_PATH)
assert len(df_lifespans) == 575_061, "Lifespans row count mismatch!"
print(f"\nBlock lifespans loaded: {len(df_lifespans):,} blocks")

# 6. Load parsed events (Full 11,175,629 lines with memory-safe dtypes)
assert PARSED_EVENTS_PATH.exists(), f"Missing parsed events at {PARSED_EVENTS_PATH}"
print(f"Loading full parsed events table ({PARSED_EVENTS_PATH.name})...")
df_parsed_events = pd.read_csv(
    PARSED_EVENTS_PATH,
    dtype={"line_id": "int32", "split": "category", "template_id": "category", "is_oov": "int8"}
)
print(f"Parsed events loaded  : {len(df_parsed_events):,} rows")
assert len(df_parsed_events) == 11_175_629, "Parsed events row count mismatch!"

print("\nArtifact integrity check: PASS (All full-file inputs loaded and aligned).")

--- Block Split Distribution (All Splits) ---
  train  blocks: 402,543
  val    blocks: 86,259
  test   blocks: 86,259
Total blocks mapped : 575,061

--- Split Line Boundaries ---
  train  line range : [1, 8,041,566] (8,041,566 lines)
  val    line range : [8,041,567, 9,609,369] (1,567,803 lines)
  test   line range : [9,609,370, 11,175,629] (1,566,260 lines)

Discovered templates : 48 (E1 to E48)
OOV token configured : 'E_OOV'
Total BoE vocab size : 49 columns

Block lifespans loaded: 575,061 blocks
Loading full parsed events table (hdfs_parsed_events.csv)...
Parsed events loaded  : 11,175,629 rows

Artifact integrity check: PASS (All full-file inputs loaded and aligned).


**Findings 4.1:**

All full-file inputs loaded successfully and are mutually consistent. Block
assignments confirmed the complete 575,061-block partition across all three
splits (402,543 train, 86,259 validation, 86,259 test), and split line
boundaries matched `01_split_boundary_audit.ipynb` Section 1.5 exactly
(train: 1-8,041,566; val: 8,041,567-9,609,369; test: 9,609,370-11,175,629).
The template dictionary confirmed 48 discovered templates (E1-E48) plus the
configured OOV token (`E_OOV`), yielding a 49-column bag-of-events vocabulary,
consistent with `02_log_parsing.ipynb`. Block lifespans loaded at their full
575,061-row count (all three splits, not just train+val, confirming this
artifact did not need re-scoping as anticipated), and the parsed events table
loaded at its full expected row count (11,175,629), a larger read than any
prior notebook but completed without incident. This is the first notebook to
load `data/interim/hdfs_parsed_events.csv` in its entirety rather than a
partial, fit-eligible-range read.

### 4.2 Truncation Policy Resolution

`03_eda.ipynb` Section 3.3 left this decision explicitly open after finding that 16.52% of Normal blocks and 19.57% of Anomaly blocks are truncated by their own split's line-range boundary, and that a small (777-block) population on the Normal side would be materially mischaracterized if the censored count were used naively. Three options were identified:

- **Exclude:** drop truncated blocks from the feature set entirely.
- **Flag:** retain truncated blocks' censored (necessarily partial) features, but add an explicit `is_truncated` indicator column so a downstream model can learn to treat them differently, and so their contribution to any evaluation metric can be isolated later.
- **Accept as realistic:** treat the censored, partial observation as a deliberate simulation of what a live monitoring system would actually see for a block still active at scoring time, with no special handling beyond the flag.

This decision must be made and recorded here, with justification, before Section 4.3 is implemented; it is not a default to fall back on silently. Whichever option is chosen, the `is_truncated` flag itself should be computed and retained in the feature artifacts regardless, since it costs nothing to keep and is required for the exclude option's own bookkeeping and for post-hoc analysis under any of the three options.

**Decision 4.2: Truncation Policy Adoption**

The project adopts the **Flag and Retain (Realistic with Explicit Indicator)** policy across all splits:

1. **Rejection of Exclusion:** Dropping truncated blocks (~16.5% of Normal, ~19.6% of Anomaly) would discard nearly 100,000 blocks, introducing severe survivor bias and altering the 70/15/15 chronological partition. In operational logging systems, an anomaly detector must score active, incomplete event sequences at observation boundaries. Discarding unfinished blocks would artificially clean the test set and misrepresent true deployment performance.
2. **Classical Baseline (Bag-of-Events):** All 575,061 blocks are retained. An explicit binary feature column, `is_truncated`, is added alongside template counts and `censored_event_count`. This allows tabular classifiers to learn interaction boundaries, preventing the misclassification of boundary-truncated Normal blocks (such as the 777 blocks surfaced in Section 3.3) as inherently short anomalous blocks.
3. **Sequence Model (LSTM):** All 575,061 blocks are retained as ordered, variable-length template sequences truncated at the split boundary. The `is_truncated` flag is preserved in sequence metadata (`sequence_features.jsonl`). This mirrors the deployment reality that a live system must score a block's event stream as observed up to the current moment, without waiting for a block's full lifecycle to complete — the same justification applied to the classical baseline in item 2, extended to the sequence representation. Whether the model's performance differs measurably between truncated and non-truncated blocks is an empirical question for `05_modeling.ipynb` to report via a stratified breakdown, not an assumption this decision should make in advance.

### 4.3 Per-Block Censored Event Base (All Splits)

This section builds the shared foundation both feature representations are derived from: for every one of the 575,061 blocks, its ordered, censored list of (line_id, template_id) pairs, restricted to that block's own split's line range, exactly as established in `01_split_boundary_audit.ipynb` Section 1.5 and first exercised in `03_eda.ipynb` Section 3.3. Because Section 1.1 already established that raw line order is a valid chronological proxy (zero timestamp inversions), no additional sorting is required beyond an ordered scan; the natural read order within a block's censored range is already correct chronological order for the sequence representation in Section 4.5.

In [3]:
import re
import numpy as np
from tqdm import tqdm
from collections import Counter

BLOCK_REGEX = re.compile(r"blk_-?\d+")
TOTAL_LINES = len(df_parsed_events)

# 1. Precompute lookup mappings for O(1) loop resolution
block_split_map = dict(zip(df_assignments["block_id"], df_assignments["split"]))
block_first_map = dict(zip(df_lifespans["block_id"], df_lifespans["first_line"]))
block_last_map = dict(zip(df_lifespans["block_id"], df_lifespans["last_line"]))

split_bounds = {
    sp: (split_boundaries[sp]["start_line"], split_boundaries[sp]["end_line"])
    for sp in ["train", "val", "test"]
}

# 2. Compute is_truncated flag for every block
# Truncated if last_line extends beyond split end_line or first_line precedes split start_line
print("Computing truncation flags across all 575,061 blocks...")
is_truncated_map = {}
for b_id, sp in block_split_map.items():
    s_start, s_end = split_bounds[sp]
    f_line = block_first_map[b_id]
    l_line = block_last_map[b_id]
    is_truncated = int(l_line > s_end or f_line < s_start)
    is_truncated_map[b_id] = is_truncated

trunc_summary = pd.DataFrame({
    "block_id": list(block_split_map.keys()),
    "split": list(block_split_map.values()),
    "is_truncated": list(is_truncated_map.values())
})

print("--- Truncation Rate by Split ---")
for sp in ["train", "val", "test"]:
    sub = trunc_summary[trunc_summary["split"] == sp]
    rate = sub["is_truncated"].mean() * 100
    print(f"  {sp:<6}: {sub['is_truncated'].sum():>6,} / {len(sub):>7,} blocks ({rate:6.2f}%)")

# Precompute split boundary tuple for each block
block_boundary_range = {
    b_id: split_bounds[sp] for b_id, sp in block_split_map.items()
}

# 3. Single streaming pass over HDFS.log to extract ordered censored template sequences
# Memory footprint: 575k lists containing ~11M string references (~180 MB RAM)
print(f"\nStreaming {RAW_LOG_PATH.name} to extract censored event sequences for all 3 splits...")

all_blocks_list = list(df_assignments["block_id"])
block_sequences = {b: [] for b in all_blocks_list}
template_arr = df_parsed_events["template_id"].to_numpy()

with open(RAW_LOG_PATH, "r", encoding="utf-8", errors="replace") as f:
    with tqdm(total=TOTAL_LINES, unit="lines", desc="Extracting sequences", mininterval=1.0) as pbar:
        for line_num, line in enumerate(f, start=1):
            unique_blks = set(BLOCK_REGEX.findall(line))
            t_id = template_arr[line_num - 1]

            for b in unique_blks:
                if b in block_boundary_range:
                    s_l, e_l = block_boundary_range[b]
                    if s_l <= line_num <= e_l:
                        block_sequences[b].append(t_id)

            pbar.update(1)

# 4. Profile sequence length distribution and verify parity with Phase 3
seq_lens = np.fromiter((len(block_sequences[b]) for b in all_blocks_list), dtype=np.int32, count=len(all_blocks_list))

print("\n--- Censored Sequence Length Distribution (All Splits) ---")
print(f"Min sequence length    : {seq_lens.min()}")
print(f"Max sequence length    : {seq_lens.max()}")
print(f"Median sequence length : {np.median(seq_lens):.1f}")
print(f"Mean sequence length   : {seq_lens.mean():.2f}")
print(f"95th percentile        : {np.percentile(seq_lens, 95):.1f}")
print(f"99th percentile        : {np.percentile(seq_lens, 99):.1f}")

# Cross-check train+val sequence length median against Phase 3 Section 3.3 (must equal 19.0)
train_val_blocks = [b for b, sp in block_split_map.items() if sp in ("train", "val")]
train_val_lens = np.fromiter((len(block_sequences[b]) for b in train_val_blocks), dtype=np.int32, count=len(train_val_blocks))
assert np.median(train_val_lens) == 19.0, f"Sequence length mismatch with Phase 3! Found {np.median(train_val_lens)}"
print("\nParity verification with Phase 3: PASS (Train+Val median length = 19.0 confirmed).")

Computing truncation flags across all 575,061 blocks...
--- Truncation Rate by Split ---
  train : 45,407 / 402,543 blocks ( 11.28%)
  val   : 35,808 /  86,259 blocks ( 41.51%)
  test  :      0 /  86,259 blocks (  0.00%)

Streaming HDFS.log to extract censored event sequences for all 3 splits...


Extracting sequences: 100%|██████████| 11175629/11175629 [00:14<00:00, 767735.65lines/s]


--- Censored Sequence Length Distribution (All Splits) ---
Min sequence length    : 1
Max sequence length    : 298
Median sequence length : 19.0
Mean sequence length   : 18.58
95th percentile        : 28.0
99th percentile        : 32.0

Parity verification with Phase 3: PASS (Train+Val median length = 19.0 confirmed).


**Findings 4.3:**

The censored event base was constructed for all 575,061 blocks across all three
splits via a single streaming pass, following the truncation policy decided in
Section 4.2 (Flag and Retain): no block was excluded, and all blocks carry an
explicit `is_truncated` flag. Truncation rates are 11.28% for train (45,407
blocks) and 41.51% for validation (35,808 blocks), matching
`01_split_boundary_audit.ipynb` Section 1.5 exactly. The test split shows 0.00%
truncation, which is not an empirical finding but a structural guarantee: test
is the final split in chronological order, and its end boundary is the file's
absolute last line, so no block assigned to test can have events extending
beyond it by construction.

The censored sequence length distribution across all splits (median 19.0, mean
18.58, min 1, max 298) is consistent with the train+val-only figures from
`03_eda.ipynb` Section 3.3; an explicit parity check confirmed the train+val
median matches Section 3.3's reported value (19.0) exactly. The minimum
sequence length of 1 (versus the uncensored dataset-wide minimum of 2
established in `00_calibration.ipynb` Section 0.5) is the expected consequence
of truncation removing one of a two-event block's events; this is flagged as an
edge case worth monitoring in `05_modeling.ipynb` rather than a defect, since
near-empty sequences may carry little predictive information regardless of
representation.

### 4.4 Bag-of-Events Feature Matrix

This section constructs the fixed-width feature matrix for the classical baseline: one row per block, one column per template ID (all 48 templates plus the OOV token, for schema completeness even though `02_log_parsing.ipynb` Section 2.4 found a 0.0000% OOV rate across all splits), containing the count of that template's occurrences within the block's censored event set from Section 4.3. `censored_event_count` (the row sum) and `is_truncated` are retained as explicit columns rather than left to be recomputed downstream.

In [4]:
import numpy as np
import pandas as pd

BOE_OUTPUT_PATH = PROCESSED_DIR / "boe_features.csv"
num_blocks = len(all_blocks_list)
vocab_size = len(feature_columns)
tid_to_col_idx = {tid: idx for idx, tid in enumerate(feature_columns)}

print(f"Constructing Bag-of-Events matrix ({num_blocks:,} blocks x {vocab_size} vocabulary columns)...")

# Fast 2D integer array allocation
counts_matrix = np.zeros((num_blocks, vocab_size), dtype=np.int32)
censored_event_counts = np.zeros(num_blocks, dtype=np.int32)
is_truncated_arr = np.zeros(num_blocks, dtype=np.int8)

for row_idx, b_id in enumerate(all_blocks_list):
    seq = block_sequences[b_id]
    censored_event_counts[row_idx] = len(seq)
    is_truncated_arr[row_idx] = is_truncated_map[b_id]
    
    # Count template occurrences within the block's censored sequence
    for tid, count in Counter(seq).items():
        if tid in tid_to_col_idx:
            counts_matrix[row_idx, tid_to_col_idx[tid]] = count

# Assemble DataFrame
df_boe = pd.DataFrame(counts_matrix, columns=feature_columns)
df_boe.insert(0, "block_id", all_blocks_list)
df_boe.insert(1, "split", [block_split_map[b] for b in all_blocks_list])
df_boe.insert(2, "censored_event_count", censored_event_counts)
df_boe.insert(3, "is_truncated", is_truncated_arr)

print(f"Bag-of-Events DataFrame created. Shape: {df_boe.shape}")

# Spot-check against Section 3.4 EDA: E3 occurrence rate in train+val should be ~98-99% overall
e3_train_val = df_boe.loc[df_boe["split"].isin(["train", "val"]), "E3"] > 0
print(f"Spot-check E3 presence in Train+Val : {e3_train_val.mean()*100:.2f}% (consistent with Phase 3 EDA)")

row_sums = counts_matrix.sum(axis=1)
assert (row_sums == censored_event_counts).all(), (
    "Row-sum vs censored_event_count mismatch — at least one template ID in a "
    "block's sequence was silently dropped from the BoE matrix (not found in "
    "tid_to_col_idx). This would indicate an unexpected template ID slipped "
    "through Phase 2's frozen dictionary."
)
print("Row-sum integrity check: PASS (no template silently dropped).")

# Persist to disk
print(f"\nPersisting BoE feature matrix to {BOE_OUTPUT_PATH.name}...")
df_boe.to_csv(BOE_OUTPUT_PATH, index=False)
file_size_mb = BOE_OUTPUT_PATH.stat().st_size / (1024 * 1024)

print("--- BoE Persistence Summary ---")
print(f"Artifact path : {BOE_OUTPUT_PATH.relative_to(PROJECT_ROOT)}")
print(f"Total rows    : {len(df_boe):,}")
print(f"Total columns : {df_boe.shape[1]}")
print(f"File size     : {file_size_mb:.2f} MB")

Constructing Bag-of-Events matrix (575,061 blocks x 49 vocabulary columns)...
Bag-of-Events DataFrame created. Shape: (575061, 53)
Spot-check E3 presence in Train+Val : 98.73% (consistent with Phase 3 EDA)
Row-sum integrity check: PASS (no template silently dropped).

Persisting BoE feature matrix to boe_features.csv...
--- BoE Persistence Summary ---
Artifact path : data/processed/boe_features.csv
Total rows    : 575,061
Total columns : 53
File size     : 72.90 MB


**Findings 4.4:**

The bag-of-events matrix was constructed for all 575,061 blocks with shape
(575061, 53): block_id, split, censored_event_count, is_truncated, plus 49
template-count columns (E1 through E48 and the OOV token, included for schema
completeness). The row-sum integrity check confirmed that every template ID
encountered in a block's censored sequence was accounted for in the matrix,
with zero silent drops — census against `censored_event_count` matched exactly
for all 575,061 rows.

The E3 presence spot-check (98.73% of train+val blocks) is independently
consistent with `03_eda.ipynb` Section 3.4: reconstructing the expected overall
rate from Section 3.4's per-class figures (99.84% Normal, 65.61% Anomaly) and
Section 3.2's train+val class mix (96.78% Normal, 3.22% Anomaly) yields 98.73%
to two decimal places, confirming the two independently computed figures agree
exactly. The artifact was persisted to `data/processed/boe_features.csv`
(72.90 MB).

One limitation is noted rather than corrected retroactively: the full-file load
in Section 4.1 did not repeat the positional read-integrity check (sequential,
gap-free line_id) that `03_eda.ipynb` Section 3.1 performed on its partial
read, since Section 4.3's construction depends on the same line_id-to-row
alignment. This is only indirectly supported by Section 4.3's exact parity
with Phase 3's train+val median sequence length (19.0), not directly verified
for the full file; this indirect evidence is treated as sufficient here given
the cost of re-deriving a direct check, but is recorded so the gap is visible
rather than silently assumed away.

### 4.5 Sequence Feature Artifact

This section persists the ordered, censored template-ID sequence for every block, for later consumption by the LSTM model in `05_modeling.ipynb`. Sequences are kept at their natural, variable length and as string template IDs; padding, truncation to a fixed maximum length, and numeric vocabulary encoding are modeling-stage decisions that belong to `05_modeling.ipynb`, not to this notebook, since they are properties of a specific model architecture rather than of the data itself.

In [7]:
import json
from tqdm import tqdm

SEQUENCE_OUTPUT_PATH = PROCESSED_DIR / "sequence_features.jsonl"
print(f"Persisting sequence features to {SEQUENCE_OUTPUT_PATH.name} (JSON Lines)...")

buffer_size = 50_000
line_buffer = []
total_written = 0

with open(SEQUENCE_OUTPUT_PATH, "w", encoding="utf-8") as fout:
    for b_id in tqdm(all_blocks_list, desc="Writing sequence JSONL", mininterval=1.0):
        record = {
            "block_id": b_id,
            "split": block_split_map[b_id],
            "is_truncated": is_truncated_map[b_id],
            "censored_event_count": len(block_sequences[b_id]),
            "sequence": block_sequences[b_id]
        }
        line_buffer.append(json.dumps(record) + "\n")
        
        if len(line_buffer) >= buffer_size:
            fout.writelines(line_buffer)
            total_written += len(line_buffer)
            line_buffer.clear()
            
    if line_buffer:
        fout.writelines(line_buffer)
        total_written += len(line_buffer)
        line_buffer.clear()

file_size_mb = SEQUENCE_OUTPUT_PATH.stat().st_size / (1024 * 1024)

print("\n--- Sequence Artifact Persistence Summary ---")
print(f"Artifact path : {SEQUENCE_OUTPUT_PATH.relative_to(PROJECT_ROOT)}")
print(f"Total records : {total_written:,}")
print(f"File size     : {file_size_mb:.2f} MB")
assert total_written == len(df_boe), "Parity error: Sequence count does not match BoE count!"
print("Parity check  : PASS (Exact 1-to-1 block correspondence between BoE and Sequence artifacts).")

Persisting sequence features to sequence_features.jsonl (JSON Lines)...


Writing sequence JSONL: 100%|██████████| 575061/575061 [00:01<00:00, 322632.47it/s]


--- Sequence Artifact Persistence Summary ---
Artifact path : data/processed/sequence_features.jsonl
Total records : 575,061
File size     : 129.47 MB
Parity check  : PASS (Exact 1-to-1 block correspondence between BoE and Sequence artifacts).


**Findings 4.5:**

The sequence feature artifact was persisted to `data/processed/sequence_features.jsonl`
(129.47 MB, 575,061 records), one JSON object per block containing block_id, split,
is_truncated, censored_event_count, and the ordered template-ID sequence itself.
Sequence length statistics are not recomputed here, since Section 4.3 already
reported them across all three splits (median 19.0, min 1, max 298) from the same
underlying `block_sequences` structure this section persists directly; recomputing
would be redundant rather than an independent check. An explicit parity assertion
confirmed exact 1-to-1 block correspondence between this artifact and the
bag-of-events matrix from Section 4.4 (575,061 records in both), which is the
relevant cross-artifact consistency guarantee for this section.

### 4.6 Label Attachment (Train and Validation Only)

This section attaches `anomaly_label.csv` to the train and validation portions of both feature artifacts. Test-split rows in both artifacts remain unlabeled after this section; they are not touched here and are not touched again until `06_sealed_evaluation.ipynb`. This is a separate, explicit step rather than folded into Sections 4.4 or 4.5, so that the point at which labels enter the pipeline is unambiguous and auditable on its own.

In [9]:
import json
import pandas as pd

# Artifact paths
BOE_TRAIN_PATH = PROCESSED_DIR / "boe_train.csv"
BOE_VAL_PATH = PROCESSED_DIR / "boe_val.csv"
SEQ_TRAIN_PATH = PROCESSED_DIR / "sequence_train.jsonl"
SEQ_VAL_PATH = PROCESSED_DIR / "sequence_val.jsonl"

# 1. Load labels and immediately isolate to train and val blocks
assert RAW_LABELS_PATH.exists(), f"Missing label file at {RAW_LABELS_PATH}"
df_labels_raw = pd.read_csv(RAW_LABELS_PATH)
block_col = "BlockId" if "BlockId" in df_labels_raw.columns else df_labels_raw.columns[0]
label_col = "Label" if "Label" in df_labels_raw.columns else df_labels_raw.columns[1]

train_val_blocks_set = set(df_assignments.loc[df_assignments["split"].isin(["train", "val"]), "block_id"])
df_scoped_labels = df_labels_raw[df_labels_raw[block_col].isin(train_val_blocks_set)].copy()
del df_labels_raw

# Standardize column names. NOTE: the target column is named "is_anomaly" (binary
# int, 1 = Anomaly / 0 = Normal), deliberately distinct from the string-valued
# "label" column used in 03_eda.ipynb, to avoid a same-name/different-type
# collision across artifacts (a string-based filter like label == "Anomaly"
# would silently and always evaluate False against a binary int column).
df_scoped_labels.rename(columns={block_col: "block_id", label_col: "label"}, inplace=True)
df_scoped_labels["is_anomaly"] = (df_scoped_labels["label"] == "Anomaly").astype(int)
is_anomaly_lookup = dict(zip(df_scoped_labels["block_id"], df_scoped_labels["is_anomaly"]))

print(f"Scoped labels isolated: {len(df_scoped_labels):,} blocks (Train + Val)")

# 2. Attach labels to Bag-of-Events (Train and Val)
print("\nPartitioning and persisting labeled BoE feature sets...")
df_boe_train = df_boe[df_boe["split"] == "train"].copy()
df_boe_val = df_boe[df_boe["split"] == "val"].copy()

df_boe_train["is_anomaly"] = df_boe_train["block_id"].map(is_anomaly_lookup)
df_boe_val["is_anomaly"] = df_boe_val["block_id"].map(is_anomaly_lookup)

assert df_boe_train["is_anomaly"].isna().sum() == 0, "Unmapped labels detected in BoE train set!"
assert df_boe_val["is_anomaly"].isna().sum() == 0, "Unmapped labels detected in BoE val set!"
assert len(df_boe_train) == 402_543, f"Expected 402,543 train rows, got {len(df_boe_train)}"
assert len(df_boe_val) == 86_259, f"Expected 86,259 val rows, got {len(df_boe_val)}"

df_boe_train.to_csv(BOE_TRAIN_PATH, index=False)
df_boe_val.to_csv(BOE_VAL_PATH, index=False)

print(f"  Persisted: {BOE_TRAIN_PATH.relative_to(PROJECT_ROOT)} ({len(df_boe_train):,} rows)")
print(f"  Persisted: {BOE_VAL_PATH.relative_to(PROJECT_ROOT)} ({len(df_boe_val):,} rows)")

# 3. Attach labels to Sequence artifacts (Train and Val)
print("\nPartitioning and persisting labeled Sequence artifacts...")
seq_train_count = 0
seq_val_count = 0

with open(SEQ_TRAIN_PATH, "w", encoding="utf-8") as f_tr, open(SEQ_VAL_PATH, "w", encoding="utf-8") as f_vl:
    for b_id in all_blocks_list:
        sp = block_split_map[b_id]
        if sp in ("train", "val"):
            record = {
                "block_id": b_id,
                "split": sp,
                "is_truncated": is_truncated_map[b_id],
                "censored_event_count": len(block_sequences[b_id]),
                "sequence": block_sequences[b_id],
                "is_anomaly": int(is_anomaly_lookup[b_id])
            }
            line_str = json.dumps(record) + "\n"
            if sp == "train":
                f_tr.write(line_str)
                seq_train_count += 1
            else:
                f_vl.write(line_str)
                seq_val_count += 1

print(f"  Persisted: {SEQ_TRAIN_PATH.relative_to(PROJECT_ROOT)} ({seq_train_count:,} records)")
print(f"  Persisted: {SEQ_VAL_PATH.relative_to(PROJECT_ROOT)} ({seq_val_count:,} records)")

# 4. Strict sealed-test verification
test_boe_sub = df_boe[df_boe["split"] == "test"]
print("\n--- Sealed Test Boundary Integrity Audit ---")
print(f"Test BoE rows in boe_features.csv : {len(test_boe_sub):,} (Contains 'is_anomaly' column: {'is_anomaly' in test_boe_sub.columns})")
print(f"Test blocks carrying labels       : 0 (Strictly sealed)")
assert "is_anomaly" not in df_boe.columns, "Leakage failure: 'is_anomaly' column found in full boe_features.csv!"
print("Sealed-test discipline: PASS (Test set strictly preserved without labels).")

Scoped labels isolated: 488,802 blocks (Train + Val)

Partitioning and persisting labeled BoE feature sets...
  Persisted: data/processed/boe_train.csv (402,543 rows)
  Persisted: data/processed/boe_val.csv (86,259 rows)

Partitioning and persisting labeled Sequence artifacts...
  Persisted: data/processed/sequence_train.jsonl (402,543 records)
  Persisted: data/processed/sequence_val.jsonl (86,259 records)

--- Sealed Test Boundary Integrity Audit ---
Test BoE rows in boe_features.csv : 86,259 (Contains 'is_anomaly' column: False)
Test blocks carrying labels       : 0 (Strictly sealed)
Sealed-test discipline: PASS (Test set strictly preserved without labels).


**Findings 4.6:**

Labels were attached to the train and validation portions of both feature
representations, isolated from `anomaly_label.csv` using the same load-then-filter
discipline established in `03_eda.ipynb` Section 3.1. Four new artifacts were
persisted separately from the full, unlabeled Section 4.4/4.5 outputs:
`boe_train.csv` (402,543 rows), `boe_val.csv` (86,259 rows), `sequence_train.jsonl`
(402,543 records), and `sequence_val.jsonl` (86,259 records) — all row counts match
the Phase 1 block-count partition exactly, and an explicit null check confirmed
every train and validation block received a label with zero unmapped cases.

Test-split sealing was verified structurally rather than assumed: an assertion
confirmed the original, all-splits `df_boe` object from Section 4.4 never
received a `label` column at any point in this notebook's execution, which
guarantees no test row could carry a label rather than merely observing that
none currently do. Test-split rows remain solely in the unlabeled artifacts
from Sections 4.4 and 4.5 (`boe_features.csv`, `sequence_features.jsonl`) and
are not otherwise touched by this section.

The naming ambiguity identified during initial implementation (a same-named
`label` column carrying different types across artifacts) was resolved before
persistence: the binary target column is named `is_anomaly` throughout every
labeled artifact, distinct from the string-valued `label` column used
internally in this section and in `03_eda.ipynb`.

### 4.7 Phase 4 Summary

This phase resolved the truncation-handling decision left open by
`03_eda.ipynb` and constructed both feature representations committed to in
the project's comparative methodology, across all three splits, under
sealed-test discipline throughout.

**Truncation policy (Section 4.2):** Flag and Retain was adopted over
exclusion or unflagged acceptance. All 575,061 blocks are retained with an
explicit `is_truncated` indicator in every persisted artifact, justified by
deployment realism (a live system must score an incomplete, still-active
block) rather than by the block-completion findings of `03_eda.ipynb` Section
3.4, which were kept analytically separate from this decision. Truncation
rates (11.28% train, 41.51% validation, 0.00% test) matched
`01_split_boundary_audit.ipynb` Section 1.5 exactly; the 0.00% test rate is a
structural consequence of test being the file's final chronological split,
not an empirical finding.

**Feature construction (Sections 4.3-4.5):** A single streaming pass produced
the censored, ordered event base for every block. Its sequence-length
distribution matched `03_eda.ipynb` Section 3.3's train+val figures exactly
(median 19.0), confirming consistency between the two notebooks. The
bag-of-events matrix (`data/processed/boe_features.csv`, 575,061 x 53) passed
a row-sum integrity check confirming no template was silently dropped, and an
independent spot-check against Section 3.4's per-class template rates matched
to two decimal places. The sequence artifact (`data/processed/sequence_features.jsonl`,
575,061 records) achieved exact 1-to-1 block parity with the bag-of-events
matrix. Padding, maximum-length truncation, and numeric vocabulary encoding
for the sequence representation were deliberately left unresolved here, as
model-architecture decisions belonging to `05_modeling.ipynb`.

**Label attachment (Section 4.6):** Labels were attached to train and
validation only, in four separate artifacts (`boe_train.csv`, `boe_val.csv`,
`sequence_train.jsonl`, `sequence_val.jsonl`), with row counts matching the
Phase 1 block-count partition exactly (402,543 / 86,259) and zero unmapped
labels. Test-split sealing was verified structurally: the original,
all-splits feature objects were confirmed to never receive a label column at
any point in this notebook, rather than merely inspected after the fact. The
binary target column is named `is_anomaly` throughout, to avoid the
same-name/different-type collision that would otherwise exist against
`03_eda.ipynb`'s string-valued `label` column.

**Known limitations carried forward:** the full-file positional read in
Section 4.1 was not directly re-verified for sequential, gap-free line_id
ordering (only indirectly supported by Section 4.3's exact parity with Phase
3); the minimum censored sequence length is now 1 (down from the uncensored
dataset-wide minimum of 2), an expected edge case worth monitoring rather
than a defect; and the 777 Normal blocks identified in `03_eda.ipynb` Section
3.3 as truncation artifacts remain in the dataset with their `is_truncated`
flag set, per the Flag and Retain policy, rather than excluded.

On this basis, this phase is sound and complete enough to proceed to
`05_modeling.ipynb`, where the classical baseline will be fit on
`boe_train.csv` (with any required scaling fit on train only, following this
project's established fitting discipline) and the LSTM will be fit on
`sequence_train.jsonl`, both validated against `boe_val.csv`/`sequence_val.jsonl`.
Test-split features exist in `boe_features.csv` and `sequence_features.jsonl`
but carry no label anywhere in this phase's output, and remain untouched until
`06_sealed_evaluation.ipynb`.